# 06 --- Position Estimation

Given integrated TDoA measurements from the Kalman filter, the final stage
of the PINGU pipeline solves for the transmitter position using a
**weighted non-linear least-squares (NLLS)** optimiser.  The solver
minimises the sum of squared TDoA residuals with a Levenberg-Marquardt
algorithm, initialised by a coarse grid search to avoid local minima.

This notebook demonstrates:
1. Setting up a multi-receiver geometry.
2. Computing true TDoAs from a known transmitter position.
3. Visualising the cost-function landscape.
4. Solving with exact and noisy measurements.
5. Confidence ellipses and uncertainty quantification.
6. The effect of receiver geometry on accuracy.

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

from pingu.locator.geometry import ReceiverGeometry
from pingu.locator.cost_functions import tdoa_residuals, tdoa_jacobian
from pingu.locator.solvers import TDoASolver
from pingu.locator.posterior import confidence_ellipse, position_uncertainty
from pingu.types import ReceiverConfig
from pingu.constants import SPEED_OF_LIGHT

# Reproducible random generator
rng = np.random.default_rng(seed=42)

print(f"Speed of light: {SPEED_OF_LIGHT:.0f} m/s")

## Receiver Geometry

We place 5 receivers in a regular pentagon with a 100 km radius.  This
provides good angular diversity for 2-D geolocation.

In [ ]:
# Build 5 receivers in a regular pentagon (100 km radius)
RADIUS = 100_000.0  # 100 km
N_RX = 5

receivers = []
for i in range(N_RX):
    angle = np.pi / 2 + 2 * np.pi * i / N_RX
    receivers.append(
        ReceiverConfig(
            id=f"RX{i}",
            latitude=40.0,
            longitude=-75.0,
            x=RADIUS * np.cos(angle),
            y=RADIUS * np.sin(angle),
        )
    )

geometry = ReceiverGeometry(receivers)
positions = geometry.get_positions()

# Plot receiver positions
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(positions[:, 0] / 1e3, positions[:, 1] / 1e3, "b^", markersize=12)
for i, r in enumerate(receivers):
    ax.annotate(
        r.id,
        (r.x / 1e3, r.y / 1e3),
        textcoords="offset points",
        xytext=(8, 8),
        fontsize=10,
    )
ax.set_xlabel("x (km)")
ax.set_ylabel("y (km)")
ax.set_title("Pentagon Receiver Array (100 km radius)")
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# Print baselines between adjacent receivers
print("Baselines between adjacent receivers:")
for i in range(N_RX):
    j = (i + 1) % N_RX
    baseline = geometry.get_baseline(i, j)
    print(f"  {receivers[i].id} -- {receivers[j].id}: {baseline / 1e3:.2f} km")

## True TDoAs from Known Position

We place a transmitter at a known location and compute the ground-truth
TDoA for every receiver pair.

In [ ]:
# True transmitter position
TRUE_TX = np.array([30_000.0, -20_000.0])  # (x, y) in metres

# Compute true TDoAs for all C(5,2) = 10 pairs
true_tdoas = geometry.compute_all_tdoas(TRUE_TX)
pair_indices = geometry.get_pair_indices()

print(f"Transmitter position: ({TRUE_TX[0]:.0f}, {TRUE_TX[1]:.0f}) m")
print(f"Number of pairs: {len(pair_indices)}")
print()
print(f"{'Pair':<12s}  {'TDoA (us)':>12s}")
print("-" * 28)
for k, (i, j) in enumerate(pair_indices):
    label = f"RX{i}-RX{j}"
    print(f"{label:<12s}  {true_tdoas[k] * 1e6:12.4f}")

## Cost Function Landscape

The NLLS cost function is the sum of squared weighted TDoA residuals.
Plotting this on a 2-D grid reveals the shape of the objective and
confirms that the global minimum coincides with the true transmitter
position.

In [ ]:
# Evaluate residual cost on a 2-D grid
GRID_HALF = 200_000.0  # +/- 200 km
N_GRID = 100
xs = np.linspace(-GRID_HALF, GRID_HALF, N_GRID)
ys = np.linspace(-GRID_HALF, GRID_HALF, N_GRID)
XX, YY = np.meshgrid(xs, ys)

# Unit weights for visualization
weights = np.ones(len(pair_indices), dtype=np.float64)

cost_map = np.zeros_like(XX)
for r in range(N_GRID):
    for c in range(N_GRID):
        candidate = np.array([XX[r, c], YY[r, c]])
        res = tdoa_residuals(
            candidate, positions, true_tdoas, pair_indices, weights
        )
        cost_map[r, c] = np.sum(res ** 2)

# Plot as filled contour with colorbar
fig, ax = plt.subplots(figsize=(10, 8))
levels = np.linspace(0, np.percentile(cost_map, 95), 50)
cf = ax.contourf(XX / 1e3, YY / 1e3, cost_map, levels=levels, cmap="viridis")
fig.colorbar(cf, ax=ax, label="Cost (sum of squared residuals)")

# Mark true TX position
ax.plot(TRUE_TX[0] / 1e3, TRUE_TX[1] / 1e3, "rx", markersize=14,
        markeredgewidth=2, label="True TX")
# Mark receivers
ax.plot(positions[:, 0] / 1e3, positions[:, 1] / 1e3, "w^",
        markersize=10, label="Receivers")

ax.set_xlabel("x (km)")
ax.set_ylabel("y (km)")
ax.set_title("TDoA Cost Function Landscape")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

# Verify minimum is near the true position
min_idx = np.unravel_index(np.argmin(cost_map), cost_map.shape)
min_x = XX[min_idx] / 1e3
min_y = YY[min_idx] / 1e3
print(f"Grid minimum at: ({min_x:.1f}, {min_y:.1f}) km")
print(f"True TX at:      ({TRUE_TX[0] / 1e3:.1f}, {TRUE_TX[1] / 1e3:.1f}) km")

## Solving with Levenberg-Marquardt

The `TDoASolver` uses `scipy.optimize.least_squares` with the LM method.
When no initial guess is provided, a coarse grid search seeds the
optimiser.  We first solve with exact (noiseless) TDoAs to verify the
solver recovers the true position nearly exactly.

In [ ]:
# Solve with exact TDoAs (no noise)
solver = TDoASolver(geometry)
weights = np.ones(len(pair_indices), dtype=np.float64)

result_exact = solver.solve(tdoas=true_tdoas, weights=weights)

est_pos = np.array([result_exact.x, result_exact.y])
error = np.linalg.norm(est_pos - TRUE_TX)

print("Exact TDoA solution:")
print(f"  Estimated position : ({result_exact.x:.2f}, {result_exact.y:.2f}) m")
print(f"  True position      : ({TRUE_TX[0]:.2f}, {TRUE_TX[1]:.2f}) m")
print(f"  Position error     : {error:.4f} m")
print(f"  Residual           : {result_exact.residual:.2e}")

In [ ]:
# Solve with noisy TDoAs
NOISE_STD = 1e-8  # 10 ns standard deviation

# Single noisy trial
noisy_tdoas = true_tdoas + rng.normal(0.0, NOISE_STD, size=len(true_tdoas))
result_noisy = solver.solve(tdoas=noisy_tdoas, weights=weights)
est_noisy = np.array([result_noisy.x, result_noisy.y])
error_noisy = np.linalg.norm(est_noisy - TRUE_TX)

print(f"Single noisy trial (noise std = {NOISE_STD * 1e9:.0f} ns):")
print(f"  Estimated position : ({result_noisy.x:.2f}, {result_noisy.y:.2f}) m")
print(f"  Position error     : {error_noisy:.2f} m")
print()

# Monte Carlo: 20 trials
N_TRIALS = 20
errors = []
for trial in range(N_TRIALS):
    noisy = true_tdoas + rng.normal(0.0, NOISE_STD, size=len(true_tdoas))
    res = solver.solve(tdoas=noisy, weights=weights)
    err = np.linalg.norm(np.array([res.x, res.y]) - TRUE_TX)
    errors.append(err)

errors = np.array(errors)
print(f"Monte Carlo ({N_TRIALS} trials):")
print(f"  Mean position error : {np.mean(errors):.2f} m")
print(f"  Std position error  : {np.std(errors):.2f} m")

## Confidence Ellipses

The solver returns a 2x2 position covariance matrix from which we can
compute a confidence ellipse.  The `confidence_ellipse()` function
returns the semi-major axis, semi-minor axis, and orientation angle;
`position_uncertainty()` provides a full summary including the CEP.

In [ ]:
# Use the last noisy solution for the ellipse
semi_major, semi_minor, angle_deg = confidence_ellipse(
    result_noisy.covariance, confidence=0.95
)
uncertainty = position_uncertainty(result_noisy, confidence=0.95)

print("95% Confidence Ellipse:")
print(f"  Semi-major axis : {semi_major:.2f} m")
print(f"  Semi-minor axis : {semi_minor:.2f} m")
print(f"  Orientation     : {angle_deg:.1f} degrees")
print(f"  95% radius      : {uncertainty['radius']:.2f} m")
print(f"  CEP (50%)       : {uncertainty['cep']:.2f} m")
print()

# Plot receivers, true position, estimate, and confidence ellipse
fig, ax = plt.subplots(figsize=(10, 10))

# Receivers
ax.plot(positions[:, 0] / 1e3, positions[:, 1] / 1e3, "b^",
        markersize=12, label="Receivers")
for r in receivers:
    ax.annotate(r.id, (r.x / 1e3, r.y / 1e3),
                textcoords="offset points", xytext=(8, 8), fontsize=9)

# True position
ax.plot(TRUE_TX[0] / 1e3, TRUE_TX[1] / 1e3, "rx", markersize=14,
        markeredgewidth=2, label="True Position")

# Estimate
ax.plot(result_noisy.x / 1e3, result_noisy.y / 1e3, "*",
        color="gold", markersize=18, markeredgecolor="black",
        markeredgewidth=0.5, label="Estimate")

# 95% confidence ellipse
ellipse_patch = Ellipse(
    xy=(result_noisy.x / 1e3, result_noisy.y / 1e3),
    width=2 * semi_major / 1e3,
    height=2 * semi_minor / 1e3,
    angle=angle_deg,
    edgecolor="darkorange",
    facecolor="orange",
    alpha=0.25,
    linewidth=1.5,
    label="95% Confidence",
)
ax.add_patch(ellipse_patch)

ax.set_xlabel("x (km)")
ax.set_ylabel("y (km)")
ax.set_title("Position Estimate with 95% Confidence Ellipse")
ax.legend(loc="upper left")
ax.set_aspect("equal", adjustable="datalim")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Effect of Geometry

Receiver geometry strongly influences TDoA accuracy.  Wider baselines
improve range resolution, while non-collinear layouts provide good
cross-range resolution.  We compare three geometries:

1. **Pentagon (100 km radius)** -- the baseline configuration.
2. **Pentagon (10 km radius)** -- a much smaller array.
3. **Collinear receivers** -- all receivers on a single line.

In [ ]:
def build_pentagon_receivers(radius: float) -> list[ReceiverConfig]:
    """Build 5 receivers in a regular pentagon with the given radius."""
    rxs = []
    for i in range(5):
        angle = np.pi / 2 + 2 * np.pi * i / 5
        rxs.append(
            ReceiverConfig(
                id=f"RX{i}",
                latitude=40.0,
                longitude=-75.0,
                x=radius * np.cos(angle),
                y=radius * np.sin(angle),
            )
        )
    return rxs


def build_collinear_receivers(spacing: float = 50_000.0) -> list[ReceiverConfig]:
    """Build 5 receivers along the x-axis."""
    rxs = []
    for i in range(5):
        rxs.append(
            ReceiverConfig(
                id=f"RX{i}",
                latitude=40.0,
                longitude=-75.0,
                x=(i - 2) * spacing,
                y=0.0,
            )
        )
    return rxs


# Test each geometry with the same noise level
NOISE_STD_GEOM = 1e-8
N_TRIALS_GEOM = 20

geometries = {
    "Pentagon 100 km": build_pentagon_receivers(100_000.0),
    "Pentagon 10 km": build_pentagon_receivers(10_000.0),
    "Collinear 50 km": build_collinear_receivers(50_000.0),
}

print(f"{'Geometry':<20s}  {'Mean Error (m)':>14s}  {'Std Error (m)':>14s}")
print("-" * 55)

for name, rxs in geometries.items():
    geo = ReceiverGeometry(rxs)
    slv = TDoASolver(geo)
    true_t = geo.compute_all_tdoas(TRUE_TX)
    w = np.ones(len(geo.get_pair_indices()), dtype=np.float64)

    errs = []
    for _ in range(N_TRIALS_GEOM):
        noisy = true_t + rng.normal(0.0, NOISE_STD_GEOM, size=len(true_t))
        res = slv.solve(tdoas=noisy, weights=w)
        err = np.linalg.norm(np.array([res.x, res.y]) - TRUE_TX)
        errs.append(err)

    errs = np.array(errs)
    print(f"{name:<20s}  {np.mean(errs):14.2f}  {np.std(errs):14.2f}")

print()
print("Wider geometry yields better accuracy.")
print("Collinear receivers suffer from poor cross-range resolution.")

## Summary

- The **weighted NLLS solver** with **Levenberg-Marquardt** optimisation
  reliably recovers the transmitter position from TDoA measurements.
- A **grid search** over the cost-function landscape provides a robust
  initial guess, avoiding local minima.
- **Confidence ellipses** derived from the Jacobian covariance quantify
  the position uncertainty and reveal the geometric dilution of precision.
- **Receiver geometry** is a dominant factor: wider baselines and
  non-collinear layouts produce significantly smaller position errors.